<a href="https://colab.research.google.com/github/Juna-Saputra/Pengolahan-Citra-Pola/blob/branch1/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Final


In [ ]:
# ============================================================
# PROJECT: DASHBOARD RESTORASI & ANALISIS FORENSIK CITRA (RGB)
# FILE: app.py (Siap Dijalankan)
# Perbaikan: Penguncian File Path JPG & Metadata DPI Fisik via Pillow
# ============================================================

import gradio as gr
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io
import math
import time

# ============================================================
# SECTION 1: UTILITY & METRICS (EVALUASI)
# ============================================================

def pil_to_cv2(pil_img):
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def cv2_to_pil(cv2_img):
    return Image.fromarray(cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))

def compute_mse(img1, img2):
    return np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)

def compute_psnr(mse):
    return float(20 * math.log10(255.0 / math.sqrt(mse))) if mse > 0 else float('inf')

def compute_ssim(img1, img2):
    C1 = (0.01 * 255)**2
    C2 = (0.03 * 255)**2
    i1, i2 = img1.astype(np.float64), img2.astype(np.float64)
    mu1, mu2 = cv2.GaussianBlur(i1, (11, 11), 1.5), cv2.GaussianBlur(i2, (11, 11), 1.5)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1 * mu2
    sigma1_sq = cv2.GaussianBlur(i1**2, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(i2**2, (11, 11), 1.5) - mu2_sq
    sigma12 = cv2.GaussianBlur(i1 * i2, (11, 11), 1.5) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return np.mean(ssim_map)

def compute_entropy(gray_img):
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256]).ravel()
    hist = hist[hist > 0] / np.sum(hist)
    return -np.sum(hist * np.log2(hist))

def evaluate_result(original_bgr, processed_bgr, label, start_time=None):
    orig_gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
    proc_gray = cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2GRAY)

    if orig_gray.shape != proc_gray.shape:
        orig_gray = cv2.resize(orig_gray, (proc_gray.shape[1], proc_gray.shape[0]), interpolation=cv2.INTER_CUBIC)

    mse = compute_mse(orig_gray, proc_gray)
    return {
        "Tahap": label,tabel
        "PSNR (dB)": round(compute_psnr(mse), 2),
        "SSIM": round(compute_ssim(orig_gray, proc_gray), 4),
        "MSE": round(mse, 2),
        "Sharpness": round(np.var(cv2.Laplacian(proc_gray, cv2.CV_64F)), 2),
        "Contrast": round(proc_gray.std(), 2),
        "Brightness": round(proc_gray.mean(), 2),
        "Entropy": round(compute_entropy(proc_gray), 2),
        "Time (s)": round(time.time() - start_time, 3) if start_time else 0.0
    }

# ============================================================
# SECTION 2: ADAPTIVE & PROCESSING FUNCTIONS
# ============================================================

def analyze_image(bgr_img):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = np.var(cv2.Laplacian(gray, cv2.CV_64F))
    noise_level = np.std(cv2.blur(gray, (3,3)) - gray)
    contrast = gray.std()
    brightness = gray.mean()
    entropy = compute_entropy(gray)

    return {
        "blur_score": blur_score,
        "noise_level": noise_level,
        "contrast": contrast,
        "brightness": brightness,
        "entropy": entropy
    }

def classify_blur(bgr_img, analysis_data):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur_score = analysis_data["blur_score"]

    if blur_score > 800: return "Tidak Signifikan (Jernih)"

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    var_x, var_y = np.var(sobelx), np.var(sobely)

    ratio = var_x / (var_y + 1e-6)
    if ratio > 2.0 or ratio < 0.5: return "Motion Blur"
    elif blur_score < 150: return "Gaussian Blur"
    elif blur_score < 400: return "Defocus Blur"
    return "Unknown Blur"

def adaptive_denoise(bgr_img, noise_level, manual_strength=None):
    strength = manual_strength if manual_strength else min(max(noise_level * 0.8, 3), 15)
    return cv2.fastNlMeansDenoisingColored(bgr_img, None, h=strength, hColor=strength, templateWindowSize=7, searchWindowSize=21)

def adaptive_deblur(bgr_img, blur_score, blur_type, iterations=1):
    result = bgr_img.copy()
    strength = 1.5 if blur_score > 300 else 2.5
    for _ in range(int(iterations)):
        gaussian = cv2.GaussianBlur(result, (0, 0), 2.0)
        result = cv2.addWeighted(result, strength, gaussian, -(strength-1.0), 0)
    return result

def gamma_correction(bgr_img, brightness, manual_gamma=None):
    gamma = manual_gamma if manual_gamma else (1.0 + (128 - brightness) / 255.0)
    gamma = max(0.5, min(gamma, 2.0))
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(bgr_img, table), gamma

def adaptive_clahe(bgr_img, contrast, manual_clip=None):
    clip = manual_clip if manual_clip else max(1.0, 4.0 - (contrast / 20.0))
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR), clip

def adaptive_sharpen(bgr_img, blur_score, manual_amount=None):
    amount = manual_amount if manual_amount else max(1.0, 3.0 - (blur_score / 300.0))
    gaussian = cv2.GaussianBlur(bgr_img, (0, 0), 3.0)
    sharpened = cv2.addWeighted(bgr_img, 1.0 + amount, gaussian, -amount, 0)
    return sharpened, amount

def edge_enhancement(bgr_img, strength=0.3):
    kernel_edge = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    edges = cv2.filter2D(bgr_img, -1, kernel_edge)
    return cv2.addWeighted(bgr_img, 1.0, edges, strength, 0)

def artifact_reduction(bgr_img, strength=10):
    return cv2.bilateralFilter(bgr_img, d=5, sigmaColor=strength, sigmaSpace=strength)

def image_upscaling(bgr_img, scale=2.0):
    h, w = bgr_img.shape[:2]
    return cv2.resize(bgr_img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LANCZOS4)

def generate_report(analysis, blur_type, gamma, clahe_clip, sharpen_amount):
    lines = [
        "📋 **LAPORAN ANALISIS FORENSIK CITRA OTOMATIS**",
        "---",
        f"🔍 **Klasifikasi Blur:** `{blur_type}`",
        f"📊 **Skor Awal:** Blur ({analysis['blur_score']:.2f}) | Noise ({analysis['noise_level']:.2f}) | Kontras ({analysis['contrast']:.2f}) | Kecerahan ({analysis['brightness']:.2f})",
        "\n⚙️ **Tindakan Adaptif Sistem:**"
    ]

    n_lvl = "Tinggi" if analysis['noise_level'] > 10 else "Sedang" if analysis['noise_level'] > 5 else "Rendah"
    c_lvl = "Rendah" if analysis['contrast'] < 40 else "Tinggi"

    lines.append(f"- **Noise {n_lvl} terdeteksi.** Sistem mengaplikasikan Denoising adaptif sesuai level noise.")
    if blur_type != "Tidak Signifikan (Jernih)":
        lines.append(f"- **Blur terdeteksi.** Sistem menerapkan Deblurring iteratif.")
    lines.append(f"- **Kecerahan disesuaikan.** Gamma otomatis diset ke `{gamma:.2f}`.")
    lines.append(f"- **Kontras {c_lvl}.** CLAHE Clip Limit diset ke `{clahe_clip:.2f}`.")
    lines.append(f"- **Penajaman Detail.** Radius Unsharp Mask dihitung dengan amount `{sharpen_amount:.2f}`.")
    lines.append("- **Reduksi Artefak:** Aktif (Mencegah efek halo & kontur palsu).")

    lines.append("\n✅ **Kesimpulan:** Citra berhasil dipulihkan secara otomatis berdasarkan karakteristik piksel spesifiknya.")
    return "\n".join(lines)

def step_rotate_rgb(bgr_img, angle_str):
    angle_map = {"Tidak Ada": None, "90°": cv2.ROTATE_90_CLOCKWISE, "180°": cv2.ROTATE_180, "270°": cv2.ROTATE_90_COUNTERCLOCKWISE}
    code = angle_map.get(angle_str, None)
    return cv2.rotate(bgr_img, code) if code is not None else bgr_img

def step_zoom_rgb(bgr_img, zoom_factor=1.0):
    if zoom_factor == 1.0: return bgr_img
    h, w, c = bgr_img.shape
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    zoomed = cv2.resize(bgr_img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    if zoom_factor > 1.0:
        start_h, start_w = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return zoomed

# ============================================================
# SECTION 3: VISUALIZATION HELPERS
# ============================================================

def plot_histogram(bgr_img, title):
    fig, ax = plt.subplots(figsize=(4, 3), facecolor='#111827')
    ax.set_facecolor('#111827')
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    ax.hist(gray_img.ravel(), bins=256, range=(0, 256), color='#14b8a6', alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold', color='#2dd4bf')
    ax.tick_params(colors='#9ca3af', labelsize=7)
    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.15)
    fig.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=90, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

def get_matrix_dataframe(bgr_img, size=50):
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    df = pd.DataFrame(gray_img[:min(size, gray_img.shape[0]), :min(size, gray_img.shape[1])])
    df.columns = [str(c) for c in df.columns]
    return df

def make_comparison_image(original_bgr, final_bgr):
    h1, w1 = original_bgr.shape[:2]
    h2, w2 = final_bgr.shape[:2]
    max_h = max(h1, h2)

    pad1, pad2 = np.zeros((max_h, w1, 3), dtype=np.uint8), np.zeros((max_h, w2, 3), dtype=np.uint8)
    pad1[:h1, :w1] = original_bgr
    pad2[:h2, :w2] = final_bgr

    divider = np.ones((max_h, 6, 3), dtype=np.uint8) * 150
    combined = np.hstack([pad1, divider, pad2])
    cv2.putText(combined, "ASLI", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(combined, "HASIL REKONSTRUKSI", (w1 + 16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return cv2_to_pil(combined)

# ============================================================
# SECTION 4: EXECUTORS (AUTO & MANUAL)
# ============================================================

def run_pipeline(input_image, rotation_choice, zoom_factor):
    """Eksekusi Mode Berantai Otomatis (DIJAGA 100% UTUH)"""
    start_t = time.time()
    if input_image is None:
        empty = Image.new("RGB", (400, 300), color=(17, 24, 39))
        empty_df = pd.DataFrame()
        return [empty]*7 + [plot_histogram(np.zeros((10,10,3), dtype=np.uint8), "Kosong")]*7 + [empty_df]*7 + [empty_df, "Silakan upload citra.", empty, empty, empty]

    bgr_input = pil_to_cv2(input_image)
    base_bgr = step_zoom_rgb(step_rotate_rgb(bgr_input, rotation_choice), float(zoom_factor))

    analysis = analyze_image(base_bgr)
    blur_type = classify_blur(base_bgr, analysis)

    img_denoise = adaptive_denoise(base_bgr, analysis["noise_level"])
    iters = 3 if blur_type == "Motion Blur" else 1
    img_deblur = adaptive_deblur(img_denoise, analysis["blur_score"], blur_type, iterations=iters)
    img_gamma, gamma_val = gamma_correction(img_deblur, analysis["brightness"])
    img_clahe, clahe_val = adaptive_clahe(img_gamma, analysis["contrast"])
    img_sharp, sharp_val = adaptive_sharpen(img_clahe, analysis["blur_score"])
    img_edge = edge_enhancement(img_sharp)
    img_recon = artifact_reduction(img_edge)

    stages = [
        (base_bgr, "1. Input Citra"),
        (img_denoise, "2. Denoising Adaptif"),
        (img_deblur, "3. Deblurring"),
        (img_clahe, "4. Gamma & CLAHE"),
        (img_sharp, "5. Sharpening Adaptif"),
        (img_edge, "6. Edge Enhancement"),
        (img_recon, "7. Rekonstruksi Akhir")
    ]

    metrics_list = [evaluate_result(base_bgr, img, title, start_t) for img, title in stages]

    out_images = [cv2_to_pil(img) for img, _ in stages]
    out_hists  = [plot_histogram(img, f"Hist: {title}") for img, title in stages]
    out_mats   = [get_matrix_dataframe(img) for img, _ in stages]
    comp_img = make_comparison_image(base_bgr, img_recon)
    report = generate_report(analysis, blur_type, gamma_val, clahe_val, sharp_val)

    return (*out_images, *out_hists, *out_mats, pd.DataFrame(metrics_list), report, comp_img, out_images[0], out_images[-1])


def run_mandiri_pipeline(input_image, deskew_val, scale_val, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, upscale_factor, dpi_val, selected_steps):
    """Mode Mandiri Kustom (Dengan Perbaikan Penguncian Metadata DPI via Simpan File Fisik)"""
    start_t = time.time()

    if input_image is None:
        empty_img = Image.new("RGB", (400, 300), color=(17, 24, 39))
        return (empty_img, empty_img, pd.DataFrame(), [], []) + tuple([gr.update(visible=False)]*12)

    bgr_img = pil_to_cv2(input_image)
    current_img = bgr_img.copy()

    states = [(current_img.copy(), "1. Input Asli")]
    step_num = 2

    if deskew_val != 0.0:
        h, w = current_img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), deskew_val, 1.0)
        current_img = cv2.warpAffine(current_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        states.append((current_img.copy(), f"{step_num}. Kemiringan ({deskew_val}°)")); step_num+=1

    if scale_val != 1.0:
        current_img = step_zoom_rgb(current_img, scale_val)
        states.append((current_img.copy(), f"{step_num}. Scaling ({scale_val}x)")); step_num+=1

    if "Denoising" in selected_steps:
        current_img = adaptive_denoise(current_img, noise_level=0, manual_strength=denoise_str)
        states.append((current_img.copy(), f"{step_num}. Denoising")); step_num+=1

    if "Deblurring" in selected_steps:
        current_img = adaptive_deblur(current_img, blur_score=0, blur_type="", iterations=deblur_iter)
        states.append((current_img.copy(), f"{step_num}. Deblurring")); step_num+=1

    if "Gamma Correction" in selected_steps:
        current_img, _ = gamma_correction(current_img, brightness=0, manual_gamma=gamma_val)
        states.append((current_img.copy(), f"{step_num}. Gamma ({gamma_val})")); step_num+=1

    if "CLAHE" in selected_steps:
        current_img, _ = adaptive_clahe(current_img, contrast=0, manual_clip=clahe_clip)
        states.append((current_img.copy(), f"{step_num}. CLAHE")); step_num+=1

    if "Sharpening" in selected_steps:
        current_img, _ = adaptive_sharpen(current_img, blur_score=0, manual_amount=sharp_amt)
        states.append((current_img.copy(), f"{step_num}. Sharpening")); step_num+=1

    if "Edge Enhance" in selected_steps:
        current_img = edge_enhancement(current_img, strength=edge_str)
        states.append((current_img.copy(), f"{step_num}. Edge Enhance")); step_num+=1

    if "Artifact Reduction" in selected_steps:
        current_img = artifact_reduction(current_img, strength=artifact_str)
        states.append((current_img.copy(), f"{step_num}. Artifact Reduction")); step_num+=1

    if "Image Upscaling" in selected_steps:
        current_img = image_upscaling(current_img, scale=upscale_factor)
        states.append((current_img.copy(), f"{step_num}. Upscaled ({upscale_factor}x)")); step_num+=1

    if "DPI Enhancement (Custom DPI)" in selected_steps:
        states.append((current_img.copy(), f"{step_num}. DPI Enhanced ({int(dpi_val)} DPI)")); step_num+=1

    # Konversi hasil akhir ke PIL Image (Mekanisme Warna Berubah Aman ke RGB)
    final_pil = cv2_to_pil(current_img)

    # PERBAIKAN INTI: Tulis file fisik ke disk menggunakan Pillow agar metadata terkunci erat
    output_file_path = "output_forensik_final.jpg"
    if "DPI Enhancement (Custom DPI)" in selected_steps:
        dpi_target = int(dpi_val)
        final_pil.save(output_file_path, format="JPEG", dpi=(dpi_target, dpi_target), quality=100)
    else:
        final_pil.save(output_file_path, format="JPEG", quality=100)

    gallery_out = [(cv2_to_pil(img), title) for img, title in states]
    hist_gallery_out = [(plot_histogram(img, title), title) for img, title in states]

    metrics = pd.DataFrame([
        evaluate_result(bgr_img, bgr_img, "Asli", start_t),
        evaluate_result(bgr_img, current_img, "Akhir Kustom", start_t)
    ])

    mat_updates = []
    for i in range(12):
        if i < len(states):
            img, title = states[i]
            df = get_matrix_dataframe(img)
            mat_updates.append(gr.update(value=df, label=title, visible=True))
        else:
            mat_updates.append(gr.update(value=None, visible=False))

    # Mengembalikan string path file fisik sebagai pengganti objek Gambar agar Gradio tidak me-reencode metadatanya
    return (output_file_path, make_comparison_image(bgr_img, current_img), metrics, gallery_out, hist_gallery_out) + tuple(mat_updates)


# ============================================================
# SECTION 5: UI BUILD
# ============================================================

CUSTOM_CSS = """
body, .gradio-container { background-color: #0b0f19 !important; color: #e2e8f0 !important; font-family: 'Segoe UI', Tahoma, sans-serif !important; }
.app-header { background: linear-gradient(135deg, #0f172a, #115e59); padding: 24px; border-radius: 14px; margin-bottom: 20px; text-align: center; border: 1px solid #14b8a6; }
.app-header h1 { margin: 0; font-size: 1.8rem; font-weight: 700; color: #2dd4bf; }
.section-label { font-size: 1rem; font-weight: 700; color: #2dd4bf; margin-bottom: 8px; border-bottom: 2px solid #0f766e; display: inline-block; }
.run-btn { background: linear-gradient(90deg, #0f766e, #14b8a6) !important; color: white !important; font-weight: 700 !important; border-radius: 10px !important; border: none !important; }
"""

def build_ui():
    with gr.Blocks(theme=gr.themes.Default(), css=CUSTOM_CSS, title="Dashboard Forensik RGB", js="() => document.body.classList.add('dark')") as demo:
        gr.HTML('<div class="app-header"><h1>🔬 Dashboard Restorasi & Analisis Forensik Citra</h1></div>')

        with gr.Tabs():
            # ========================================================
            # TAB 1: MODE BERANTAI (TETAP SAMA, TIDAK DIRUSAK)
            # ========================================================
            with gr.TabItem("⛓️ Mode Berantai (Otomatis)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        input_image = gr.Image(type="pil", label="Unggah Citra Buram", height=220)
                        rotation_choice = gr.Dropdown(["Tidak Ada", "90°", "180°", "270°"], value="Tidak Ada", label="🔄 Rotasi")
                        zoom_slider = gr.Slider(1.0, 4.0, step=0.5, value=1.0, label="🔍 Zoom")
                        run_btn = gr.Button("▶ Jalankan Auto-Pipeline Terpadu", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        comparison_output = gr.Image(label="Evaluasi (Sebelum vs Sesudah)", height=320)

                with gr.Tabs():
                    with gr.TabItem("🔄 Tahapan Pipeline (7 Tahap)"):
                        with gr.Row():
                            t_in = gr.Image(label="1. Input", height=200)
                            t_den = gr.Image(label="2. Denoising", height=200)
                            t_deb = gr.Image(label="3. Deblurring", height=200)
                            t_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            t_sha = gr.Image(label="5. Sharpening", height=200)
                            t_edg = gr.Image(label="6. Edge Enhance", height=200)
                            t_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("📊 Histogram"):
                        with gr.Row():
                            h_in = gr.Image(label="1. Input", height=200)
                            h_den = gr.Image(label="2. Denoising", height=200)
                            h_deb = gr.Image(label="3. Deblurring", height=200)
                            h_cla = gr.Image(label="4. CLAHE", height=200)
                        with gr.Row():
                            h_sha = gr.Image(label="5. Sharpening", height=200)
                            h_edg = gr.Image(label="6. Edge Enhance", height=200)
                            h_rec = gr.Image(label="7. Rekonstruksi", height=200)

                    with gr.TabItem("🔢 Matriks Piksel"):
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 1. Input"); m_in = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 2. Denoising"); m_den = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 3. Deblurring"); m_deb = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 4. CLAHE"); m_cla = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 5. Sharpening"); m_sha = gr.Dataframe(max_height=200)
                            with gr.Column(): gr.Markdown("### 6. Edge Enhance"); m_edg = gr.Dataframe(max_height=200)
                        with gr.Row():
                            with gr.Column(): gr.Markdown("### 7. Rekonstruksi Akhir"); m_rec = gr.Dataframe(max_height=200)

                    with gr.TabItem("📈 Laporan Analisis"):
                        metrics_table = gr.Dataframe(label="Tabel Evaluasi Kualitas Visual")
                        forensic_text = gr.Markdown(label="Kesimpulan Forensik")

                    with gr.TabItem("📥 Output Final"):
                        with gr.Row():
                            final_orig = gr.Image(label="Citra Asli")
                            final_out = gr.Image(label="Citra Hasil Restorasi")

                run_btn.click(
                    fn=run_pipeline,
                    inputs=[input_image, rotation_choice, zoom_slider],
                    outputs=[
                        t_in, t_den, t_deb, t_cla, t_sha, t_edg, t_rec,
                        h_in, h_den, h_deb, h_cla, h_sha, h_edg, h_rec,
                        m_in, m_den, m_deb, m_cla, m_sha, m_edg, m_rec,
                        metrics_table, forensic_text, comparison_output, final_orig, final_out
                    ]
                )

            # ========================================================
            # TAB 2: MODE MANDIRI (FIXED DOWNLOAD METADATA DPI)
            # ========================================================
            with gr.TabItem("🛠️ Mode Mandiri (Kustom)"):
                with gr.Row():
                    with gr.Column(scale=1):
                        img_in_m = gr.Image(type="pil", label="Unggah Citra Asli", height=200)

                        gr.HTML('<div class="section-label">Geometri</div>')
                        deskew_m = gr.Slider(-45.0, 45.0, step=1.0, value=0.0, label="📐 Koreksi Kemiringan")
                        scale_m = gr.Slider(0.1, 4.0, step=0.1, value=1.0, label="📏 Scaling Dasar")

                        gr.HTML('<div class="section-label">Parameter Proses (Termasuk Final)</div>')
                        denoise_str = gr.Slider(1.0, 30.0, step=1.0, value=10.0, label="Noise Strength")
                        deblur_iter = gr.Slider(1, 10, step=1, value=2, label="Deblur Iterations")
                        gamma_val = gr.Slider(0.1, 3.0, step=0.1, value=1.0, label="Gamma Value")
                        clahe_clip = gr.Slider(1.0, 10.0, step=0.5, value=2.0, label="CLAHE Clip Limit")
                        sharp_amt = gr.Slider(0.1, 5.0, step=0.1, value=1.5, label="Sharpen Amount")
                        edge_str = gr.Slider(0.1, 1.0, step=0.1, value=0.3, label="Edge Strength")
                        artifact_str = gr.Slider(1.0, 50.0, step=1.0, value=10.0, label="Artifact Reduct Strength")
                        upscale_factor = gr.Slider(1.0, 4.0, step=0.5, value=2.0, label="Upscale Factor (x)")
                        dpi_val = gr.Slider(72, 600, step=1, value=300, label="Set Metadata DPI Kustom")

                        steps_m = gr.CheckboxGroup(
                            ["Denoising", "Deblurring", "Gamma Correction", "CLAHE", "Sharpening", "Edge Enhance", "Artifact Reduction", "Image Upscaling", "DPI Enhancement (Custom DPI)"],
                            label="Terapkan Efek Secara Berurutan"
                        )
                        btn_m = gr.Button("▶ Eksekusi Mode Mandiri", elem_classes="run-btn")

                    with gr.Column(scale=2):
                        with gr.Tabs():
                            with gr.TabItem("👁️ Pratinjau"):
                                comp_out_m = gr.Image(label="Komparasi (Asli vs Akhir)")
                                # Mengunci visual keluaran ke tipe format jpeg
                                img_out_m = gr.Image(label="Output Visual Akhir", format="jpeg")

                            with gr.TabItem("📊 Evaluasi Metrik"):
                                metrics_out_m = gr.Dataframe(label="Tabel Evaluasi Akhir")

                            with gr.TabItem("🎬 Citra Tracker"):
                                gallery_m = gr.Gallery(label="Visual Tahapan", columns=4, object_fit="contain")

                            with gr.TabItem("📈 Histogram Tracker"):
                                hist_gallery_m = gr.Gallery(label="Histogram Tahapan", columns=4, object_fit="contain")

                            with gr.TabItem("🔢 Matriks Tracker"):
                                mat_m_boxes = []
                                with gr.Row():
                                    for i in range(4):
                                        with gr.Column(): mat_m_boxes.append(gr.Dataframe(visible=False, max_height=200))
                                with gr.Row():
                                    for i in range(4, 8):
                                        with gr.Column(): mat_m_boxes.append(gr.Dataframe(visible=False, max_height=200))
                                with gr.Row():
                                    for i in range(8, 12):
                                        with gr.Column(): mat_m_boxes.append(gr.Dataframe(visible=False, max_height=200))

                btn_m.click(
                    fn=run_mandiri_pipeline,
                    inputs=[img_in_m, deskew_m, scale_m, denoise_str, deblur_iter, gamma_val, clahe_clip, sharp_amt, edge_str, artifact_str, upscale_factor, dpi_val, steps_m],
                    outputs=[img_out_m, comp_out_m, metrics_out_m, gallery_m, hist_gallery_m] + mat_m_boxes
                )

    return demo

if __name__ == "__main__":
    demo = build_ui()
    demo.launch(inline=True, share=True, debug=True)